In [1]:
# ================================
# INSTALLS
# ================================

!pip install pymupdf pillow pytesseract -q
!apt-get install tesseract-ocr -q

# ================================
# IMPORTS
# ================================

import re
import fitz
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import pytesseract
import os
from google.colab import drive

drive.mount("/content/drive")

# ================================
# CONFIG
# ================================

PDF_DIR   = "/content/drive/MyDrive/glaze/swatches/3_pdfs"
SAVE_BASE = "/content/drive/MyDrive/glaze/swatches/type_3_new"

VARIANCE_THRESHOLD = 21

SOLO_W_MIN,   SOLO_W_MAX   = 270, 320
SOLO_H_MIN,   SOLO_H_MAX   = 330, 380
DOUBLE_W_MIN, DOUBLE_W_MAX = 560, 610
DOUBLE_H_MIN, DOUBLE_H_MAX = 330, 380

SWATCH_TOP    = 0.0
SWATCH_BOTTOM = 0.32
SWATCH_LEFT   = 0.55
SWATCH_RIGHT  = 1.0

TEXT_TOP    = 0.37
TEXT_BOTTOM = 0.45
TEXT_LEFT   = 0.63
TEXT_RIGHT  = 0.97

INSET = 8

# Page number → cone folder name
CONE_MAP = {
    1: "cone_6",
    2: "cone_10",
}

# ================================
# HELPER FUNCTIONS
# ================================

def colour_variance(crop):
    return float(np.std(crop.astype(float)))


def overlap(a, b, threshold=0.5):
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    ix = max(0, min(ax+aw, bx+bw) - max(ax, bx))
    iy = max(0, min(ay+ah, by+bh) - max(ay, by))
    inter = ix * iy
    smaller = min(aw * ah, bw * bh)
    return inter / smaller > threshold if smaller > 0 else False


def detect_cards(img_array):
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    edges = cv2.Canny(blurred, 30, 100)
    kernel = np.ones((3, 3), np.uint8)
    dilated = cv2.dilate(edges, kernel, iterations=2)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    accepted_solo   = []
    accepted_double = []
    rejected        = []

    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w < 20 or h < 20:
            continue

        crop = img_array[y:y+h, x:x+w]
        variance = colour_variance(crop)

        is_solo   = (SOLO_W_MIN   <= w <= SOLO_W_MAX   and SOLO_H_MIN   <= h <= SOLO_H_MAX)
        is_double = (DOUBLE_W_MIN <= w <= DOUBLE_W_MAX and DOUBLE_H_MIN <= h <= DOUBLE_H_MAX)

        if variance <= VARIANCE_THRESHOLD:
            rejected.append((x, y, w, h, variance, f"flat var={variance:.0f}", crop))
        elif is_solo:
            accepted_solo.append((x, y, w, h, variance))
        elif is_double:
            accepted_double.append((x, y, w, h, variance))
        else:
            rejected.append((x, y, w, h, variance, f"size {w}x{h}", crop))

    def dedup(lst):
        filtered = []
        for c in sorted(lst, key=lambda c: c[2]*c[3], reverse=True):
            if not any(overlap(c[:4], kept[:4]) for kept in filtered):
                filtered.append(c)
        return sorted(filtered, key=lambda c: (c[1], c[0]))

    return dedup(accepted_solo), dedup(accepted_double), rejected


def detect_swatches_in_cell(img_array):
    MIN_SIZE = 80
    MAX_SIZE = 90

    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    edges = cv2.Canny(blurred, 30, 100)
    kernel = np.ones((3, 3), np.uint8)
    dilated = cv2.dilate(edges, kernel, iterations=2)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    accepted = []
    rejected = []

    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w < 20 or h < 20:
            continue

        crop = img_array[y:y+h, x:x+w]
        variance = colour_variance(crop)

        reasons = []
        if not (MIN_SIZE <= w <= MAX_SIZE and MIN_SIZE <= h <= MAX_SIZE):
            reasons.append(f"size {w}x{h}")
        if abs(w - h) > 10:
            reasons.append(f"not square (diff={abs(w-h)})")
        if variance <= VARIANCE_THRESHOLD:
            reasons.append(f"flat var={variance:.0f}")

        if reasons:
            rejected.append((x, y, w, h, variance, ", ".join(reasons), crop))
        else:
            accepted.append((x, y, w, h, variance))

    filtered = []
    for c in sorted(accepted, key=lambda c: c[2]*c[3], reverse=True):
        if not any(overlap(c[:4], kept[:4]) for kept in filtered):
            filtered.append(c)

    return sorted(filtered, key=lambda c: (c[1], c[0])), rejected


def clean_ocr(ocr_text, fallback):
    lines = [l.strip() for l in ocr_text.strip().splitlines() if l.strip()]

    SW_PREFIX = re.compile(r'^SW-\d+$', re.IGNORECASE)

    cleaned = []
    for line in lines:
        if SW_PREFIX.match(line):
            cleaned.append(line.strip())
        else:
            word = line.strip().lower().replace(" ", "-")
            word = "".join(c for c in word if c.isalnum() or c == "-")
            if word:
                cleaned.append(word)

    result = "-".join(cleaned)
    return result if result else fallback


def unique_filename(save_dir, base_name):
    filename = f"{base_name}.png"
    counter = 1
    while os.path.exists(os.path.join(save_dir, filename)):
        filename = f"{base_name}_{counter}.png"
        counter += 1
    return filename


# ================================
# MAIN — PROCESS ALL PDFs
# ================================

pdf_files = [
    f for f in sorted(os.listdir(PDF_DIR))
    if f.lower().endswith(".pdf")
]

print(f"Found {len(pdf_files)} PDFs to process\n")

for pdf_name in pdf_files:
    pdf_path = os.path.join(PDF_DIR, pdf_name)
    original_filename = os.path.splitext(pdf_name)[0]

    print(f"\n{'='*60}")
    print(f"Processing: {pdf_name}")
    print(f"{'='*60}")

    doc = fitz.open(pdf_path)

    # ---- BLOCK 1: DETECT CARDS PER PAGE ----
    # Store cards grouped by page: {page_number: [(crop, ...), ...]}
    cards_by_page = {}

    for page_number, page in enumerate(doc, start=1):
        pix = page.get_pixmap(dpi=150)
        img_array = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)

        solo, double, rejected = detect_cards(img_array)
        print(f"  Page {page_number} — {len(solo)} solo, {len(double)} double, {len(rejected)} rejected")

        single_cards = []

        for x, y, w, h, var in solo:
            single_cards.append(img_array[y:y+h, x:x+w])

        for x, y, w, h, var in double:
            crop = img_array[y:y+h, x:x+w]
            mid = crop.shape[1] // 2
            single_cards.append(crop[:, :mid])
            single_cards.append(crop[:, mid:])

        cards_by_page[page_number] = single_cards
        print(f"    → {len(single_cards)} single cards")

    # ---- BLOCK 2: PROCESS EACH PAGE INTO ITS CONE FOLDER ----
    for page_number, single_cards in cards_by_page.items():
        cone_folder = CONE_MAP.get(page_number, f"page_{page_number}")
        SAVE_DIR = os.path.join(SAVE_BASE, original_filename, cone_folder)
        os.makedirs(SAVE_DIR, exist_ok=True)

        print(f"\n  Page {page_number} → {cone_folder} ({len(single_cards)} cards)")

        total_swatches = 0
        total_saved    = 0

        for card_idx, card in enumerate(single_cards):
            card_h, card_w = card.shape[:2]

            # Swatch corner
            sy1 = int(card_h * SWATCH_TOP)
            sy2 = int(card_h * SWATCH_BOTTOM)
            sx1 = int(card_w * SWATCH_LEFT)
            sx2 = int(card_w * SWATCH_RIGHT)
            corner = card[sy1:sy2, sx1:sx2]

            # OCR region
            ty1 = int(card_h * TEXT_TOP)
            ty2 = int(card_h * TEXT_BOTTOM)
            tx1 = int(card_w * TEXT_LEFT)
            tx2 = int(card_w * TEXT_RIGHT)
            text_region = card[ty1:ty2, tx1:tx2]

            ocr_text = pytesseract.image_to_string(
                Image.fromarray(text_region),
                config="--psm 6"
            ).strip()

            swatches, _ = detect_swatches_in_cell(corner)

            for swatch_idx, (x, y, w, h, variance) in enumerate(swatches):
                total_swatches += 1

                swatch = corner[y+INSET:y+h-INSET, x+INSET:x+w-INSET]

                fallback  = f"page{page_number:02d}_card{card_idx+1:02d}_unknown"
                base_name = clean_ocr(ocr_text, fallback)
                filename  = unique_filename(SAVE_DIR, base_name)

                Image.fromarray(swatch).save(os.path.join(SAVE_DIR, filename))
                total_saved += 1
                print(f"      Saved: {filename}")

        print(f"    Swatches detected : {total_swatches}")
        print(f"    Files saved       : {total_saved}")
        print(f"    Files in folder   : {len(os.listdir(SAVE_DIR))}")

    doc.close()

print(f"\n{'='*60}")
print("All PDFs processed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 35.4 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Mounted at /content/drive
Found 12 PDFs to process


Processing: SW-108-Green-Tea.pdf
  Page 1 — 6 solo, 3 double, 10 rejected
    → 12 single cards
  Page 2 — 6 solo, 3 double, 10 rejected
    → 12 single cards

  Page 1 → cone_6 (12 cards)
      Saved: black-walnut.png
      Saved: pri-blue.png
      Saved: honeycomb.png
      Saved: lava-rock.png
      Saved: lime-shower.png
      Saved: dark-flux.png
      Saved: frost-blue.png
      Saved: dunes.png
      Saved: sea-salt.png
      Saved: cordovan.png
      Saved: indigo-rain.png
      Saved: leather.png
    Swatches detected : 12
    Files saved       : 12
    Files in folder   : 12

  Page 2 → cone_10 (12 cards)
      Saved: black-walnut.png
      